In [1]:
# RAG
# 우리회사의 복지제도는?
# LLM은 학습데이터에 없는 최신/특정 정보를 모름

# RAG 해결책  (Tetrieval Augmented Generation) = 검색 + 생성
# 1. 회사 문서에서 관련 정보 검색
# 2. 검색된 정보를 LLM에게 컨텍스트로 제공
# 3. llm이 컨텍스 기반으로 정확한 답변 생성

# [준비 단계]
# 문서들 ->청크분할->벡터변환->벡터DB저장

#[쿼리 단계]
# 질문->벡터변환->유사도검색->상위 k개 선택 -> 컨텍스트 +질문->LLM->답변

In [ ]:
%pip install llama-index

In [1]:
from llama_index.core import Document, VectorStoreIndex

In [9]:
from google.colab import userdata
userdata.get('OPENAI_API_KEY')[:5]

'sk-pr'

In [10]:
# 1 문서 준비
import openai
import os
openai.api_key = userdata.get('OPENAI_API_KEY')
document = [
    Document(text='대한민국의 수도는 서울입니다.'),
    Document(text="프랑스의 수도는 파리 입니다.")
]
# 2 인덱스 생성(자동으로 벡터화)
# 각 청크를 openai api 로 벡터화
# 인메모리방식으로 벡터 스토어에 저장
index = VectorStoreIndex.from_documents(document)

# 3 쿼리 엔진 생성
query_engine = index.as_query_engine(similarity_top_k=1)

# 4 쿼리 실행
response = query_engine.query("대한민국의 수도는 어디입니까?")
print(response)

서울


In [11]:
# 청크 : 문서검색의 최소단위 모델이 한번에 처리할수 있는 길이로 잘라낸 텍스트
# 모델 입력 길이 제한, 문서가 길면 한번에 처리 할 수 없어서 청크로 나눠 처리
# 벡터 DB에서 문서전체가 아니라 청크단위로 벡터화
# 질문과 유사한 작은 단위를 찾아 답변을 생성
# 전체문서를 이해하는 대신 청크별로 처리해서  중요한 부분에 집중

# 작을수록 : 정확한 검색, 많은 api 호출
# 클수록 : 넓은 컨텍스트, 적은 api 호출
from llama_index.core import Settings
Settings.chunk_size = 512  # 기본값
# Settings.chunk_overlap = 128  # 기본값
Settings.chunk_overlap = 50 # 청크 간 겹침

# 유사도 임계값 설정
from llama_index.core.postprocessor import SimilarityPostprocessor
query_engine = index.as_query_engine(
    similarity_top_k=2,  # 유사도 상위 2
    node_postprocessors=[
        SimilarityPostprocessor(similarity_cutoff=0.7)  # 유사도 0.7미만의 문서는 제외 (노이즈 제거)
    ]
)
# 배치 처리
Settings.embed_batch_size = 100

한국어 데이터로 RAG 구현

In [12]:
# 한국어 데이터로 RAG 구현
from llama_index.core import Document,VectorStoreIndex,Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

In [13]:
#. 1. LLM, 임베딩 모델 설정
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0.1)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

In [14]:
# 2. 문서 준비
documents = [
    Document(
        text="김치는 한국의 대표적인 발효 음식입니다. 배추에 고춧가루, 마늘, 생강 등을 넣어 만듭니다.",
        metadata={"source": "한국 음식 백과", "category": "반찬"}
    ),
    Document(
        text="비빔밥은 밥 위에 여러 가지 나물과 고기, 계란을 올려 고추장과 섞어 먹는 음식입니다.",
        metadata={"source": "한국 음식 백과", "category": "밥 요리"}
    ),
    Document(
        text="불고기는 양념한 소고기를 구워 먹는 한국의 전통 음식입니다. 달콤하고 짭짤한 맛이 특징입니다.",
        metadata={"source": "한국 음식 백과", "category": "고기 요리"}
    ),
    Document(
        text="떡볶이는 가래떡에 고추장 양념을 넣어 볶은 한국의 길거리 음식입니다. 달콤하고 매운 맛이 특징입니다.",
        metadata={"source": "한국 음식 백과", "category": "분식"}
    ),
]

In [15]:
# 3. 벡터 인덱스 생성
index = VectorStoreIndex.from_documents(documents)

In [16]:
# 4. 쿼리 엔진 생성
query_engine = index.as_query_engine(
    similarity_top_k=2,
    # node_postprocessors=[SimilarityPostprocessor(similarity_cutoff=0.7)]
)

In [17]:
# 5. 질문하기
questions = [
    '김치는 어떤 음식인가요?',
    '비빔밥을 어떻게 먹나요?',
    '한국의 고기 요리에는 뭐가 있나요?'
]
for q in questions:
  response = query_engine.query(q)
  print(f'질문:{q} 답변 :{response}')

질문:김치는 어떤 음식인가요? 답변 :김치는 한국의 대표적인 발효 음식으로, 배추에 고춧가루, 마늘, 생강 등을 넣어 만들어집니다.
질문:비빔밥을 어떻게 먹나요? 답변 :비빔밥은 밥 위에 여러 가지 나물과 고기, 계란을 올린 후 고추장과 섞어 먹습니다.
질문:한국의 고기 요리에는 뭐가 있나요? 답변 :한국의 고기 요리 중 하나로 불고기가 있습니다. 불고기는 양념한 소고기를 구워 먹는 전통 음식으로, 달콤하고 짭짤한 맛이 특징입니다.
